# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [12]:
# Write your code below.
%load_ext dotenv
%dotenv


In [13]:
import dask.dataframe as dd


+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [20]:
import os
from glob import glob

price_files = glob(os.path.join(os.getenv('PRICE_DATA'), '**/*.parquet'),recursive=True)
price_dd = dd.read_parquet(price_files)


For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [ ]:
# Write your code below.

# Add lags for variables Close and Adj_Close:

dd_feat = price_dd.groupby('ticker').apply(
    lambda df: df.assign(
        Close_lag = df['Close'].shift(1),
        Adj_Close_lag = df['Adj Close'].shift(1)
    ),
)

# Add returns based on close and hi_lo_range columns
dd_feat = dd_feat.assign(
    returns=lambda df: df['Close'] / df['Close_lag'] - 1,
    hi_lo_range=lambda df: df['High'] - df['Low']
)

dd_feat.compute()



C:\Users\faizk\AppData\Local\Temp\ipykernel_33772\3434274816.py:5: UserWarning: `meta` is not specified, inferred from partial data. Please provide `meta` if the result is unexpected.
  Before: .apply(func)
  After:  .apply(func, meta={'x': 'f8', 'y': 'f8'}) for dataframe result
  or:     .apply(func, meta=('x', 'f8'))            for series result
  dd_feat = price_dd.groupby('ticker').apply(


Date        Open        High         Low       Close  \
ticker                                                                     
SYNH   230927 2018-01-02   43.900002   44.400002   43.099998   44.099998   
       230928 2018-01-03   44.049999   44.299999   43.150002   43.799999   
       230929 2018-01-04   43.200001   43.200001   40.500000   41.299999   
       230930 2018-01-05   41.349998   41.599998   40.500000   40.500000   
       230931 2018-01-08   39.500000   46.849998   39.500000   41.250000   
...                  ...         ...         ...         ...         ...   
ACN    141198 2016-12-23  117.400002  118.209999  117.110001  117.480003   
       141199 2016-12-27  117.360001  118.480003  117.209999  117.550003   
       141200 2016-12-28  118.000000  118.000000  116.089996  116.610001   
       141201 2016-12-29  116.980003  117.980003  116.510002  117.010002   
       141202 2016-12-30  117.559998  117.949997  116.589996  117.129997   

                Adj Close     Volume    source ticker  Year   Close_lag  \
ticker                                                                    
SYNH   230927   44.099998  1067000.0  SYNH.csv   SYNH  2018         NaN   
       230928   43.799999   631600.0  SYNH.csv   SYNH  2018   44.099998   
       230929   41.299999  1958300.0  SYNH.csv   SYNH  2018   43.799999   
       230930   40.500000  1484300.0  SYNH.csv   SYNH  2018   41.299999   
       230931   41.250000   489800.0  SYNH.csv   SYNH  2018   40.500000   
...                   ...        ...       ...    ...   ...         ...   
ACN    141198  111.277794  1697200.0   ACN.csv    ACN  2016  117.790001   
       141199  111.344101  1546300.0   ACN.csv    ACN  2016  117.480003   
       141200  110.453720  1797700.0   ACN.csv    ACN  2016  117.550003   
       141201  110.832603  1633900.0   ACN.csv    ACN  2016  116.610001   
       141202  110.946266  1739500.0   ACN.csv    ACN  2016  117.010002   

               Adj_Close_lag   returns  hi_lo_range  
ticker                                               
SYNH   230927            NaN       NaN     1.300003  
       230928      44.099998 -0.006803     1.149998  
       230929      43.799999 -0.057078     2.700001  
       230930      41.299999 -0.019370     1.099998  
       230931      40.500000  0.018519     7.349998  
...                      ...       ...          ...  
ACN    141198     111.571434 -0.002632     1.099998  
       141199     111.277794  0.000596     1.270004  
       141200     111.344101 -0.007997     1.910004  
       141201     110.453720  0.003430     1.470001  
       141202     110.832603  0.001026     1.360001  

[330869 rows x 14 columns]

+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [29]:
# Write your code below.
import pandas as pd

dd_feat_pd = pd.DataFrame(dd_feat.compute())

dd_feat_pd['10_day_mean'] = dd_feat_pd['returns'].rolling(window=10).mean()

dd_feat_pd



c:\Users\faizk\miniconda3\envs\dsi_participant\lib\site-packages\dask\dataframe\groupby.py:210: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return g.apply(func, *args, **kwargs)


Date        Open        High         Low       Close  \
ticker                                                                     
SYNH   230135 2014-11-07   20.100000   20.750000   19.610001   20.490000   
       230136 2014-11-10   20.299999   21.600000   20.100000   20.670000   
       230137 2014-11-11   20.700001   21.490000   20.330000   20.959999   
       230138 2014-11-12   20.750000   21.250000   20.549999   20.900000   
       230139 2014-11-13   20.840000   21.459999   20.840000   21.350000   
...                  ...         ...         ...         ...         ...   
ACN    141952 2019-12-24  210.710007  211.669998  210.559998  211.610001   
       141953 2019-12-26  211.610001  212.119995  211.259995  212.050003   
       141954 2019-12-27  212.449997  212.630005  211.610001  212.220001   
       141955 2019-12-30  212.240005  212.240005  209.050003  210.639999   
       141956 2019-12-31  209.979996  211.059998  209.440002  210.570007   

                Adj Close     Volume    source ticker  Year   Close_lag  \
ticker                                                                    
SYNH   230135   20.490000  4853100.0  SYNH.csv   SYNH  2014         NaN   
       230136   20.670000   246700.0  SYNH.csv   SYNH  2014   20.490000   
       230137   20.959999   224700.0  SYNH.csv   SYNH  2014   20.670000   
       230138   20.900000   144500.0  SYNH.csv   SYNH  2014   20.959999   
       230139   21.350000   210100.0  SYNH.csv   SYNH  2014   20.900000   
...                   ...        ...       ...    ...   ...         ...   
ACN    141952  210.795059   998500.0   ACN.csv    ACN  2019  210.830002   
       141953  211.233368  1059800.0   ACN.csv    ACN  2019  211.610001   
       141954  211.402710  1292200.0   ACN.csv    ACN  2019  212.050003   
       141955  209.828781  1193900.0   ACN.csv    ACN  2019  212.220001   
       141956  209.759064  1371000.0   ACN.csv    ACN  2019  210.639999   

               Adj_Close_lag   returns  hi_lo_range  10_day_mean  
ticker                                                            
SYNH   230135            NaN       NaN     1.139999          NaN  
       230136      20.490000  0.008785     1.500000          NaN  
       230137      20.670000  0.014030     1.160000          NaN  
       230138      20.959999 -0.002863     0.700001          NaN  
       230139      20.900000  0.021531     0.619999          NaN  
...                      ...       ...          ...          ...  
ACN    141952     210.018051  0.003700     1.110001     0.004750  
       141953     210.795059  0.002079     0.860001     0.004750  
       141954     211.233368  0.000802     1.020004     0.004029  
       141955     211.402710 -0.007445     3.190002     0.001941  
       141956     209.828781 -0.000332     1.619995     0.002711  

[330869 rows x 15 columns]

Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?
+ Would it have been better to do it in Dask? Why?

Not neccesary to convert to pandas as the task can be done in Dask. But pandas library is more tuned for tasks of this nature. 
(1 pt)

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.